[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/05_gradient_descent.ipynb)

# Bài 05 — Gradient Descent

**🎯 Mục tiêu**
- Hiểu và tự cài đặt gradient descent (GD) cho hàm 1 biến và nhiều biến.
- Thấy tận mắt ảnh hưởng của **learning rate**: quá nhỏ, vừa, quá lớn (phân kỳ).
- Hiểu vì sao bài toán **"méo"** (ill-conditioned) làm GD zig-zag và vì sao **chuẩn hoá dữ liệu** giúp hội tụ nhanh.
- Biết các tiêu chí dừng, cực tiểu địa phương, và giới thiệu momentum, SGD/mini-batch.

GD sẽ được dùng để train Linear Regression (chương 06), Logistic Regression (07), Neural Network (27+).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import plot_loss_contour, set_style

set_style()

## 1. Ý tưởng

Muốn tìm $\theta$ làm $f(\theta)$ nhỏ nhất. Đứng ở điểm hiện tại, nhìn **độ dốc** và bước một bước nhỏ **xuống dốc**:

$$\theta_{t+1} = \theta_t - \eta\,\nabla f(\theta_t)$$

- $\eta$ (learning rate, AIVN ký hiệu $\alpha$): độ dài bước.
- Gần cực tiểu, gradient nhỏ dần → bước tự động nhỏ lại.

Cài đặt from scratch cho hàm một biến $f(x) = (x - 3)^2 + 1$, $f'(x) = 2(x - 3)$:

In [ ]:
def gradient_descent(grad, x0, lr, n_iters=50):
    xs = [np.asarray(x0, dtype=float)]
    for _ in range(n_iters):
        xs.append(xs[-1] - lr * grad(xs[-1]))
    return np.array(xs)


f = lambda x: (x - 3) ** 2 + 1
df = lambda x: 2 * (x - 3)
path = gradient_descent(df, x0=-2.0, lr=0.1, n_iters=30)
print("x sau 30 bước:", path[-1].round(5), "| f(x) =", f(path[-1]).round(6))

## 2. Ảnh hưởng của learning rate

In [ ]:
xs = np.linspace(-4, 10, 300)
settings = [(0.02, "quá nhỏ: hội tụ chậm"), (0.3, "vừa phải"), (0.95, "lớn: dao động"), (1.05, "quá lớn: PHÂN KỲ")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (lr, title) in zip(axes, settings):
    p = gradient_descent(df, x0=-2.0, lr=lr, n_iters=15)
    ax.plot(xs, f(xs), color="gray")
    ax.plot(p, f(p), "o-", color="red", ms=4)
    ax.set_ylim(0, 60)
    ax.set_title(f"η = {lr} — {title}")
plt.show()

Với $f(x) = (x-3)^2 + 1$, ta có $x_{t+1} - 3 = (1 - 2\eta)(x_t - 3)$: hội tụ khi $|1 - 2\eta| < 1 \Leftrightarrow 0 < \eta < 1$; nhanh nhất khi $\eta = 0.5$; dao động khi $0.5 < \eta < 1$; phân kỳ khi $\eta > 1$. Tổng quát, learning rate tối đa phụ thuộc vào **độ cong** (Hessian) của hàm.

Thực hành tốt: vẽ **loss theo vòng lặp**. Loss tăng hoặc nhảy loạn → giảm $\eta$; loss giảm rất chậm → tăng $\eta$. Thử các giá trị theo thang log: 0.001, 0.003, 0.01, 0.03, 0.1, ...

In [ ]:
fig, ax = plt.subplots()
for lr in [0.01, 0.03, 0.1, 0.3]:
    p = gradient_descent(df, x0=-2.0, lr=lr, n_iters=60)
    ax.plot(f(p) - 1, label=f"η = {lr}")
ax.set_yscale("log")
ax.set_xlabel("vòng lặp")
ax.set_ylabel("f(x) − f(x*)")
ax.legend()
ax.set_title("Loss theo vòng lặp (log scale)")
plt.show()

## 3. Nhiều biến — bài toán "méo" và zig-zag

$f(w, b) = w^2 + \kappa\,b^2$. Khi $\kappa$ lớn, các đường đồng mức là elip rất dẹt: gradient gần như vuông góc với hướng tới cực tiểu → GD **zig-zag**, và learning rate bị giới hạn bởi chiều cong nhất ($\eta < 1/\kappa$) nên chiều còn lại đi rất chậm.

In [ ]:
def run(kappa, lr, n_iters=40, start=(-4.0, 2.0)):
    grad = lambda t: np.array([2 * t[0], 2 * kappa * t[1]])
    return gradient_descent(grad, np.array(start), lr, n_iters)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, kappa, lr in [(axes[0], 1.0, 0.3), (axes[1], 15.0, 0.06)]:
    path = run(kappa, lr)
    plot_loss_contour(lambda w, b: w**2 + kappa * b**2, (-5, 5), (-3, 3), path=path, ax=ax)
    ax.set_title(f"κ = {kappa}: sau 40 bước cách cực tiểu {np.linalg.norm(path[-1]):.3f}")
plt.show()

**Vì sao chuẩn hoá feature giúp GD?** Trong Linear Regression, nếu một feature có thang đo hàng nghìn (diện tích m²) và feature khác thang đo đơn vị (số phòng), loss surface sẽ rất "méo" như hình bên phải. Chuẩn hoá về cùng thang đo làm loss surface "tròn" hơn → GD hội tụ nhanh hơn nhiều. Ta sẽ kiểm chứng ở chương 04 và 06.

## 4. Tiêu chí dừng

Trong thực tế không chạy vô hạn. Dừng khi:
1. đạt số vòng lặp tối đa;
2. $\lVert\nabla f\rVert < \varepsilon$ (gradient đủ nhỏ);
3. loss giảm không đáng kể: $|f(\theta_{t}) - f(\theta_{t+1})| < \varepsilon$;
4. (ML) loss trên **validation** ngừng giảm → *early stopping* (chương 28).

In [ ]:
def gd_with_stopping(f, grad, x0, lr, tol=1e-8, max_iters=10_000):
    x = np.asarray(x0, dtype=float)
    for it in range(max_iters):
        g = grad(x)
        if np.linalg.norm(g) < tol:
            return x, it, "‖∇f‖ < tol"
        x = x - lr * g
    return x, max_iters, "hết số vòng lặp"


for lr in [0.01, 0.1, 0.4]:
    x, it, reason = gd_with_stopping(f, df, -2.0, lr)
    print(f"η={lr:<5} dừng sau {it:5d} vòng ({reason}), x = {x:.6f}")

## 5. Hàm không lồi: cực tiểu địa phương

Với hàm không lồi, điểm GD hội tụ tới **phụ thuộc điểm khởi tạo**:

In [ ]:
g = lambda x: x**4 - 3 * x**2 + 0.6 * x
dg = lambda x: 4 * x**3 - 6 * x + 0.6
xs = np.linspace(-2.2, 2.2, 400)
fig, ax = plt.subplots()
ax.plot(xs, g(xs), color="gray")
for x0, color in [(-2.0, "tab:blue"), (0.3, "tab:red"), (2.0, "tab:green")]:
    p = gradient_descent(dg, x0, lr=0.02, n_iters=60)
    ax.plot(p, g(p), "o-", ms=3, color=color, label=f"x0 = {x0} → {p[-1]:.2f}, f = {g(p[-1]):.2f}")
ax.legend()
ax.set_title("Khởi tạo khác nhau → cực tiểu khác nhau")
plt.show()

Neural network có loss không lồi, nhưng thực tế cho thấy với nhiều tham số, hầu hết cực tiểu địa phương "đủ tốt"; vấn đề lớn hơn thường là **điểm yên ngựa** (saddle point) và vùng phẳng — momentum và Adam (chương 28) giúp vượt qua chúng.

## 6. Momentum — "quả bóng có quán tính" (giới thiệu)

$$\mathbf v_{t+1} = \beta\mathbf v_t + \nabla f(\theta_t), \qquad \theta_{t+1} = \theta_t - \eta\,\mathbf v_{t+1}$$

Momentum tích luỹ hướng đi nhất quán và triệt tiêu dao động qua lại → giảm zig-zag:

In [ ]:
def momentum_gd(grad, x0, lr, beta=0.9, n_iters=40):
    x, v = np.asarray(x0, dtype=float), np.zeros_like(np.asarray(x0, dtype=float))
    path = [x.copy()]
    for _ in range(n_iters):
        v = beta * v + grad(x)
        x = x - lr * v
        path.append(x.copy())
    return np.array(path)


kappa = 15.0
grad = lambda t: np.array([2 * t[0], 2 * kappa * t[1]])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, path) in zip(axes, [("GD", gradient_descent(grad, np.array([-4.0, 2.0]), 0.06, 40)),
                                   ("GD + momentum (β=0.8)", momentum_gd(grad, [-4.0, 2.0], 0.02, beta=0.8))]):
    plot_loss_contour(lambda w, b: w**2 + kappa * b**2, (-5, 5), (-3, 3), path=path, ax=ax)
    ax.set_title(f"{name}: khoảng cách cuối = {np.linalg.norm(path[-1]):.4f}")
plt.show()

## 7. Batch, Stochastic và Mini-batch GD (giới thiệu)

Trong ML, loss là **trung bình trên $n$ mẫu**: $L(\theta) = \frac1n\sum_i \ell_i(\theta)$, nên gradient cũng là trung bình của $n$ gradient thành phần. Khi $n$ rất lớn (hàng triệu mẫu), tính gradient đầy đủ mỗi bước rất tốn kém:

| Biến thể | Mỗi bước dùng | Đặc điểm |
|---|---|---|
| Batch GD | toàn bộ $n$ mẫu | gradient chính xác, mỗi bước đắt |
| Stochastic GD (SGD) | 1 mẫu ngẫu nhiên | rất rẻ, nhiễu nhiều |
| Mini-batch GD | $B$ mẫu (32–512) | cân bằng; tận dụng vectorization/GPU — chuẩn trong deep learning |

Chương 06 (bài *Batch, SGD, Mini-batch GD* — từ khoá AIVN) cài đặt và so sánh chi tiết 3 biến thể trên Linear Regression.

## 🧠 Kiểm tra nhanh

1. Với $f(x) = 5x^2$, learning rate lớn nhất để GD còn hội tụ là bao nhiêu?
2. Loss tăng dần qua các vòng lặp — nguyên nhân khả dĩ nhất và cách sửa?
3. Vì sao chuẩn hoá feature giúp gradient descent hội tụ nhanh hơn?
4. Momentum giúp gì trong bài toán có condition number lớn?

## 📌 Tóm tắt

- GD: $\theta \leftarrow \theta - \eta\nabla f(\theta)$; learning rate quyết định hội tụ/phân kỳ.
- Bài toán méo (condition number lớn) → zig-zag, chậm; chuẩn hoá dữ liệu và momentum giúp ích.
- Hàm lồi: GD tìm cực tiểu toàn cục; hàm không lồi: phụ thuộc khởi tạo.
- Thực tế dùng mini-batch SGD + momentum/Adam.

## 📚 Tài liệu tham khảo

- Gabriel Goh — *Why Momentum Really Works* (Distill, 2017), https://distill.pub/2017/momentum/
- Vũ Hữu Tiệp — [Gradient Descent (phần 1, 2)](https://machinelearningcoban.com/2017/01/12/gradientdescent/)